<a href="https://colab.research.google.com/github/karelin2006/downloadertorrentongoogledisk/blob/main/googleFilesDownloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import subprocess
import sys

magnet_link = input("Вставьте вашу магнет-ссылку: ").strip()
save_path = "/content/drive/MyDrive/Downloads"

if not magnet_link:
    print("Ошибка: Ссылка пустая!")
else:
    cmd = [
        "aria2c",
        "--summary-interval=1",
        "--seed-time=0",

        # --- Блок хеширования и докачки ---
        "-c",                               # Включает режим продолжения загрузки (continue)
        "--check-integrity=true",           # Принудительно проверяет хеш каждого блока имеющихся файлов
        "--conditional-get=true",
        "--auto-file-renaming=false",       # Запрещает создавать дубликаты вида file.1.mkv
        "--allow-overwrite=true",           # Разрешает модифицировать существующие недокачанные файлы

        # --- Стабильность на Google Диске ---
        "--file-allocation=none",           # Отключает аллокацию (спасает от сбоев FUSE/Google Drive)
        "--disk-cache=32M",                 # Кэширует операции в RAM, снижая нагрузку на сеть Drive

        # --- Метаданные и пиры ---
        "--bt-save-metadata=true",          # Сохраняет .torrent, чтобы при следующем сбое не тянуть магнет заново
        "--enable-dht=true",
        "--enable-peer-exchange=true",

        "-d", save_path,
        magnet_link
    ]

    print("\n[1/2] Получение метаданных и проверка существующих файлов (хеширование)...")
    print("Это может занять пару минут, если на Диске уже гигабайты данных.\n")

    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in iter(process.stdout.readline, ''):
            line_clean = line.strip()

            # Отображение процесса хеширования и прогресса скачивания
            if any(marker in line_clean for marker in ["Checksum", "checking", "Verifying"]):
                print(f"[Проверка] {line_clean}")
            elif "[" in line_clean and "]" in line_clean and ("%" in line_clean or "DL:" in line_clean):
                sys.stdout.write(f"\r{line_clean}")
                sys.stdout.flush()
            elif any(k in line_clean for k in ["[NOTICE]", "[ERROR]", "[WARN]"]):
                print(f"\n{line_clean}")

        process.wait()

        if process.returncode == 0:
            print("\n\nВсё недостающее успешно докачано!")
        else:
            print(f"\n\nПроцесс завершился с кодом ошибки: {process.returncode}")

    except Exception as e:
        print(f"\nОшибка при запуске: {e}")

Вставьте вашу магнет-ссылку: magnet:?xt=urn:btih:639ad7b62a65566ac61273d025daaee66eca6eca&dn=rutor.info_%D0%9E%D1%87%D0%B5%D0%BD%D1%8C+%D1%81%D1%82%D1%80%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5+%D0%B4%D0%B5%D0%BB%D0%B0+%2F+Stranger+Things+%5BS01-04%5D+%282016-2022%29+BDRip+1080p%2C+WEB-DLRip+1080p+%7C+D%2C+P&tr=udp://opentor.net:6969&tr=http://retracker.local/announce

[1/2] Получение метаданных и проверка существующих файлов (хеширование)...
Это может занять пару минут, если на Диске уже гигабайты данных.

[#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B][Проверка] [#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B] [Checksum:#60a477 0B/39GiB(0%)]
[#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B][Проверка] [#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B] [Checksum:#60a477 152MiB/39GiB(0%)]
[#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B][Проверка] [#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B] [Checksum:#60a477 266MiB/39GiB(0%)]
[#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B][Проверка] [#60a477 0B/39GiB(0%) CN:0 SD:0 DL:0B] [Checksum:#60a477 432MiB/39GiB(1%)]
[#60a